# Saliency Maps & Grad-CAM

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Saliency maps look at $|\partial f / \partial x|$ to highlight which input pixels matter. Grad-CAM extends this to *activation* gradients, producing a smoother heatmap localised to objects.


## Mathematical Formulation

Vanilla saliency:
$$S(x) = \Big|\frac{\partial f_c(x)}{\partial x}\Big|$$

Grad-CAM (last conv layer with activations $A^k$):
$$\alpha^k_c = \text{GAP}\!\left(\frac{\partial f_c}{\partial A^k}\right),\quad L_c = \text{ReLU}\!\Big(\sum_k \alpha^k_c\,A^k\Big)$$


## Implementation


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
def vanilla_saliency(model, x, target):
    x = x.clone().detach().requires_grad_(True)
    logits = model(x)
    logits[0, target].backward()
    return x.grad.abs().squeeze().detach()

def grad_cam(model, conv_layer, x, target):
    activations, gradients = {}, {}
    def fwd_hook(_, __, out): activations['v'] = out
    def bwd_hook(_, gin, gout): gradients['v'] = gout[0]
    h1 = conv_layer.register_forward_hook(fwd_hook)
    h2 = conv_layer.register_full_backward_hook(bwd_hook)
    try:
        logits = model(x)
        model.zero_grad(); logits[0, target].backward()
        A = activations['v']; G = gradients['v']
        alpha = G.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((alpha * A).sum(1, keepdim=True))
        return cam.squeeze().detach()
    finally:
        h1.remove(); h2.remove()


## Experiment


In [ ]:
# Toy CNN on random input
torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 3)
)
x = torch.randn(1, 1, 16, 16)
print('saliency shape:', vanilla_saliency(cnn, x, target=0).shape)
print('grad-cam shape :', grad_cam(cnn, cnn[2], x, target=0).shape)


## Discussion

- Vanilla saliency is noisy in the input space; smooth it (SmoothGrad averages saliency over noisy inputs).
- Grad-CAM works only with the last convolutional feature map.
- For transformers/ViTs use *Attention Rollout* or *Grad-CAM on patch tokens*.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
